In [1]:

dbutils.widgets.text("catalog", "", "Catalog Name")
dbutils.widgets.text("schema", "", "Schema Name")
dbutils.widgets.text("warehouse", "", "Warehouse")
dbutils.widgets.text("govern_table", "", "AI Governance output table")

Box(children=(Label(value='Catalog Name'), Text(value='')))

Box(children=(Label(value='Schema Name'), Text(value='')))

Box(children=(Label(value='Warehouse'), Text(value='')))

Box(children=(Label(value='AI Governance output table'), Text(value='')))

In [2]:
from uuid import uuid4

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
warehouse = dbutils.widgets.get("warehouse")
govern_table = dbutils.widgets.get("govern_table")

run_id = f"{uuid4()}"

In [3]:
from governer.utils import get_genie_workspace
import datetime

genie_space_title="Genie Governer Space"

spark.sql(f"use catalog {catalog}")
spark.sql(f"use schema {schema}")
tables = [f"{catalog}.{schema}.{table.name}" for table in spark.catalog.listTables()]

genie_space = get_genie_workspace(genie_space_title, warehouse, tables)


In [4]:
from governer.utils import AiResponse
from databricks.sdk.service.dashboards import GenieSpace
from governer.utils import task_genie

def govern_table_comment(table: str, space: GenieSpace) -> AiResponse:
    table_comment_govern_prompt = f"""
        You are a data governance assistant for a Databricks workspace.

        Your task is to evaluate the existing comment for the table named {table}
        Use the table metadata, schema, column names, and any available sample data to determine whether the current comment is sufficient.
        Do not attempt to execute your recommendation or any data or metadata modifications.

        A table comment is considered SUFFICIENT only if it meets ALL of the following criteria:
        1. Clearly describes the business purpose of the table.
        2. Explains what type of data the table contains.
        3. Identifies the main entity or subject represented by the table.
        4. Provides context for how the table is typically used (analytics, reporting, ingestion stage, etc.).
        5. Is not empty, vague, or purely technical (e.g., "table for storing data", "generated table", etc.).

        If the comment meets all these criteria, respond with exactly:
        <response_example>
        SUFFICIENT
        <response_example>

        If the comment is missing, empty, or does not meet the criteria above, generate a concise but informative replacement comment based on the table schema and observed data.

        Your response must follow these strict rules:
        - Return ONLY a single SQL statement.
        - The SQL statement must add or update the table comment.
        - The SQL must be directly executable in a Databricks notebook.
        - Do not include explanations, markdown, formatting, or additional text.
        
        Example format:

        <response_example>
        COMMENT ON TABLE {table} IS 'Clear business-oriented description of the table and its purpose.';
        <response_example>
    """
    return task_genie(space, table_comment_govern_prompt)


In [5]:
def evaluate_table_comment_governance_response(space: GenieSpace, table: str, response: AiResponse) -> float | None :
    evaluation_prompt = f"""
        You are a data governance validation assistant for a Databricks workspace.

        Your task is to evaluate the correctness and compliance of the output produced by another model that analyzes table comments.

        Inputs provided to you:
        - table name is {table}: the table being evaluated
        - the exact output produced by the first model is:
        <first_model_response>
        "{response.text}"
        </first_model_response>

        You are allowed to inspect:
        - the table metadata
        - the table schema
        - the existing table comment
        - table properties and sample data if available
        
        You are not allowed to modify data or execute any altering statements.

        Your goal is to estimate the probability that the first model's output is valid according to the rules below.

        The first model was required to follow these rules:
        <first_model_prompt>
        {response.prompt}
        </first_model_prompt>
        Scoring:
        Return a probability from 0.0 to 1.0 representing your confidence that the output of the first model is correct and valid.

        Guidelines:
        - 1.0 = perfectly valid and compliant
        - 0.8-0.99 = minor issues but mostly correct
        - 0.5-0.79 = partially valid or questionable
        - 0.2-0.49 = major issues
        - 0.0-0.19 = clearly invalid

        Output rules:
        - Return ONLY a single number between 0.0 and 1.0
        - Do not include explanations
        - Do not include text
        - Do not include formatting
        - Example valid outputs:
        <example_output>
        0.92
        </example_output>
        <example_output>
        0.31
        </example_output>
        <example_output>
        1.0
        </example_output>
    """
    result = task_genie(space, evaluation_prompt)
    if not result.success:
        return None
    try:
        return float(result.text)
    except Exception:
        return None
     


In [6]:
from governer.utils import get_genie_workspace, GovernStatus, trash_genie_workspace
from governer.schemas.governance.table_comment_actions import table_comment_actions

res = []

space = get_genie_workspace(space_title=genie_space_title, warehouse_id=warehouse, tables=tables)

for table in tables:
    result = govern_table_comment(genie_space, table)
    eval = evaluate_table_comment_governance_response(genie_space, table, result)
    res.append({
         "run_id": run_id, 
         "table_name": table, 
         "govern_success": result.success, 
         "govern_error": result.error,
         "evaluation": eval,
         "sql": result.text, 
         "status": GovernStatus.GENERATED.name, 
         "timestamp": datetime.datetime.now()
     })

trash_genie_workspace(space=space)

govern = spark.createDataFrame(data=res,schema=table_comment_actions)
govern.write.mode("append").saveAsTable(govern_table)